In [1]:
import torch
from torch import nn
import timm
import json
from HybridNAS import HybridNAS
from CompressedViT import CompressedViT
from Pruning.PruneUtils import compute_grads
from torchvision import transforms
from torchvision.datasets import ImageFolder

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
with open("best_architecture.json", "r") as f:
    search_dict = json.load(f)

num_classes = 200
model = timm.create_model("vit_small_patch16_224", pretrained=False, num_classes=num_classes)
checkpoint = torch.load("D:\\Tesi\\FirstFineTuning\\best_model.pth")
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [3]:
pruned_model = HybridNAS.apply_pruning(state=search_dict, model=model)
comp_model = CompressedViT(search_dict, pruned_model)

In [4]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


orig_params = count_params(pruned_model)
new_params = count_params(comp_model)
print(f"Originale: {orig_params / 1e6:.2f}M")
print(f"Compresso: {new_params / 1e6:.2f}M")
print(f"Riduzione: {100 * (orig_params - new_params) / orig_params:.2f}%")

Originale: 21.74M
Compresso: 19.02M
Riduzione: 12.51%


In [5]:
data_config = timm.data.resolve_model_data_config(model)
imagenet_mean, imagenet_std = data_config["mean"], data_config["std"]

search_transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(mean=imagenet_mean, std=imagenet_std)])

path = "D:\\Tesi\\Sets\\Set1\\val"
batch_size = 128

search_set = ImageFolder(root=path, transform=search_transform)
search_loader = torch.utils.data.DataLoader(search_set, batch_size=batch_size, shuffle=False, num_workers=1, pin_memory=True)
pruned_model = pruned_model.to(device)
pruned_model.eval()
_, final_accuracy = compute_grads(pruned_model, nn.CrossEntropyLoss(), device, search_loader)
print(f"accuracy prima della compressione: {final_accuracy}")
comp_model = comp_model.to(device)
comp_model.eval()
_, final_accuracy = compute_grads(comp_model, nn.CrossEntropyLoss(), device, search_loader)
print(f"accuracy dopo la compressione: {final_accuracy}")

accuracy prima della compressione: 0.5617308937823834
accuracy dopo la compressione: 0.5442033678756477
